In [29]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile
from src.constants import Constants as C
from pathlib import Path
import torchmetrics


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
from src.parsers import parse_phonemes, wav_to_logmel, windows_and_labels
from src.parsers import PhonemeWindowDataset
from src.parsers import PhonemeWindowDataset2
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, save_checkpoint
from src.augment import augment_audio, SpecAugment

In [31]:
DATA_DIR2 = "../AutorskieDane/AutorskiDataset"

In [32]:
dataset = PhonemeWindowDataset2(
    DATA_DIR2, max_files=6700, verbose=True,
    standardize=True, silences_same=True,
    augment_audio=augment_audio,
)
print("X shape:", dataset.X.shape, "y shape:", dataset.y.shape)

found 46 TextGrid files under ../AutorskieDane/AutorskiDataset


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:41: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


total windows: 45035
          S: 741
          Z: 348
          a: 3114
          b: 538
          c: 838
          d: 617
         dZ: 6
         dz: 77
        dzj: 230
          e: 3393
        eo5: 26
          f: 382
          g: 381
          h: 448
          i: 1280
         i2: 1117
          j: 818
          k: 1148
          l: 610
          m: 1160
          n: 1202
         n~: 846
          o: 2397
        oc5: 434
          p: 1033
          r: 309
          s: 1321
         sj: 743
        sil: 13584
         sp: 0
          t: 1397
         tS: 433
        tsj: 565
          u: 1211
          v: 1158
          w: 431
          z: 647
         zj: 45
        oov: 7
X shape: torch.Size([45035, 128, 8]) y shape: torch.Size([45035])


In [33]:
n_val = max(1, int(0.2 * len(dataset)))
n_tr = len(dataset) - n_val
train_set, val_set = random_split(
    dataset, [n_tr, n_val],
    generator=torch.Generator().manual_seed(0)
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=8)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CRNN().to(device)
print("params:", sum(p.numel() for p in model.parameters()))

params: 1252103


In [ ]:
optim = torch.optim.NAdam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
EPOCHS = 15
metric = torchmetrics.Accuracy(task="multiclass", num_classes=C.N_CLASSES).to(device)


history = train_model(
    model,
    optim,
    crit,
    metric,
    train_loader,
    val_loader,
    n_epochs=EPOCHS,
    patience=2,
    factor=0.5,
    device=device,
)

/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:24